# E36 共情年代研究：公开表与有限斜率复算

2026-10-30。仅核三表粒度、样本/年份/权重、指定缺失与2008—2018的带截距加权直线点估计。没有运行SPSS、所有三次项/协变量模型、p值/置信区间、异常值敏感性或总体研究复现；不输出原行。累积调查参与人数不是回归行数。

来源：官方目录 https://api.osf.io/v2/nodes/ygebu/files/osfstorage/ ；Study 1 https://osf.io/download/dntb5/ ，Study 2a https://osf.io/download/d9ejf/ ，Study 2b https://osf.io/download/tvdb2/ 。下载并以原文件名放在工作目录；需numpy与pyreadstat及依赖。原SAV不上传到知识库。输入SHA-256逐一检查，详细值在代码。

作者语法的REGWGT表示WLS，NOORIGIN保留截距，参见IBM原文：https://www.ibm.com/docs/en/spss-statistics/30.0.0?topic=regression-regwgt-subcommand-command 与 https://www.ibm.com/docs/en/spss-statistics/30.0.0?topic=regression-origin-noorigin-subcommands-command 。此处NumPy复算限定点估计，不能说作者语法已经完整执行。

下列输出来自实际普通Python进程顺序运行，execution_count为null；未作Jupyter内核/界面验收。独立加权协方差公式仅核同一指定斜率的数值一致性，不代表模型假设验证。未使用Validation Dataset或补充PD/FS语法。


In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pyreadstat
sources = {
 "1": ("Study 1 data to share.sav", "620dc9f985d46f87108328584973fc9ed844ce346f71eee2e9ddb529cd6c99ee", "data.year", "num.ps", ["pt", "ec"]),
 "2a": ("Study 2a data to share.sav", "f8235b0bc5f4bd99046d103ac45cc60f7f4bb75a4045fa2780e2ce575825bfaa", "year", "num.participants", ["perspective_taking"]),
 "2b": ("Study 2b data to share.sav", "6c46a6a9cf0367fa21cb98a38bb0adf2ef4e162e840dc6995752cee0a18a0c46", "year", "num.participants", ["empathic_concern"])
}
results = {}
for tag, (filename, digest, year, weight, outcomes) in sources.items():
    path = Path(filename)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest
    d, meta = pyreadstat.read_sav(path, apply_value_formats=False)
    rec = {"rows": len(d), "columns": len(d.columns), "year_min": float(d[year].min()), "year_max": float(d[year].max()), "unique_years": int(d[year].nunique()), "participant_weight_sum_all_rows": float(d[weight].sum()), "outcomes": {}}
    assert d[year].notna().all() and d[weight].notna().all() and d[weight].gt(0).all()
    for ycol in outcomes:
        valid = d[ycol].notna()
        sub = {"nonmissing_rows": int(valid.sum()), "participant_weight_sum_nonmissing_rows": float(d.loc[valid, weight].sum())}
        if tag != "1":
            sub["missing_outcome_years"] = [int(v) for v in d.loc[~valid, year]]
        t = d.loc[d[year].ge(2008), [year, ycol, weight]].dropna()
        x, y, w = (t[c].to_numpy() for c in [year, ycol, weight])
        X = np.column_stack([np.ones(len(x)), x - x.mean()])
        assert np.linalg.matrix_rank(X) == 2
        coef = np.linalg.lstsq(X * np.sqrt(w)[:, None], y * np.sqrt(w), rcond=None)[0]
        mx, my = np.average(x, weights=w), np.average(y, weights=w)
        vx, vy = np.average((x-mx)**2, weights=w), np.average((y-my)**2, weights=w)
        slope_check = np.average((x-mx)*(y-my), weights=w) / vx
        assert np.isclose(coef[1], slope_check, rtol=1e-9, atol=1e-12)
        sub["2008_2018_point_check_only"] = {"aggregate_rows": len(t), "participant_weight_sum": float(w.sum()), "slope_per_year": float(coef[1]), "weighted_standardized_slope": float(coef[1] * np.sqrt(vx / vy))}
        rec["outcomes"][ycol] = sub
    if tag == "1":
        rec["missing_demographics_before_after_mean_substitution"] = {c: int(d[c].isna().sum()) for c in ["percent.male", "percent.maleMEANSUB", "percent.white", "percent.whiteMEANSUB"]}
        rec["stored_square_max_abs_error"] = float(np.max(np.abs(d["c.data.year.squared"] - d["c.data.year"]**2)))
        rec["stored_cube_max_abs_error"] = float(np.max(np.abs(d["c.data.year.cubed"] - d["c.data.year"]**3)))
    results[tag] = rec
print(json.dumps(results, ensure_ascii=False, indent=2))


{
  "1": {
    "rows": 198,
    "columns": 54,
    "year_min": 1979.0,
    "year_max": 2018.0,
    "unique_years": 33,
    "participant_weight_sum_all_rows": 38693.0,
    "outcomes": {
      "pt": {
        "nonmissing_rows": 172,
        "participant_weight_sum_nonmissing_rows": 31978.0,
        "2008_2018_point_check_only": {
          "aggregate_rows": 111,
          "participant_weight_sum": 20176.0,
          "slope_per_year": 0.02442673381443903,
          "weighted_standardized_slope": 0.3096120613809328
        }
      },
      "ec": {
        "nonmissing_rows": 181,
        "participant_weight_sum_nonmissing_rows": 35462.0,
        "2008_2018_point_check_only": {
          "aggregate_rows": 116,
          "participant_weight_sum": 22755.0,
          "slope_per_year": 0.023367568539337042,
          "weighted_standardized_slope": 0.2448220277989296
        }
      }
    },
    "missing_demographics_before_after_mean_substitution": {
      "percent.male": 11,
      "percent.male